In [4]:
import data.tromso_data as td
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots
import o2_fev1_analysis.plot_helpers as ph
import data.helpers as dh
import data.breathe_data as bd
import models.var_builders as var_builders
import pandas as pd

In [2]:
dfBR = bd.load_meas_from_excel("BR_O2_FEV1_FEF2575_conservative_smoothing_with_idx")

In [2]:
df = td.build_meas_df("T5")

INFO:root:Processing Age
INFO:root:Processing Sex
INFO:root:Processing Height



*** Building O2 Saturation and FEV1 dataframe ***


INFO:root:Processing FEV1
INFO:root:Processing FEF2575
INFO:root:Processing O2 Saturation


In [9]:
df.to_excel(f"{dh.get_path_to_main()}/ExcelFiles/TR/TR5_O2_FEV1_FEF2575_20250428.xlsx")

In [6]:
df["FEF2575%FEV1"] = df.FEF2575 / df.FEV1 * 100
df["Date Recorded"] = "01-01-2001"
df["Date Recorded"] = pd.to_datetime(df["Date Recorded"]).dt.date

# Add idx
(
    HFEV1,
    uFEV1,
    ecFEV1,
    AR,
    HO2Sat,
    O2SatFFA,
    IA,
    UO2Sat,
    O2Sat,
    ecFEF2575prctecFEV1,
    Var_ar_change,
) = var_builders.o2sat_fev1_fef2575_long_model_noise_shared_healthy_vars_and_temporal_ar(
    160,
    40,
    "Male",
    ar_change_cpt_suffix="_shape_factor_single_laplace_1.6",
    ecfev1_noise_model_suffix="_std_add_mult_ecfev1",
    fef2575_cpt_suffix="",
)
df[f"idx {ecFEV1.name}"] = df.apply(
    lambda row: ecFEV1.get_bin_idx_for_value(row["FEV1"]), axis=1
)
df[f"idx {ecFEF2575prctecFEV1.name}"] = df.apply(
    lambda row: ecFEF2575prctecFEV1.get_bin_idx_for_value(row["FEF2575%FEV1"]),
    axis=1,
)
df[f"idx {O2Sat.name}"] = df.apply(
    lambda row: O2Sat.get_bin_idx_for_value(row["O2 Saturation"]), axis=1
)

In [8]:
df.to_excel(f"{dh.get_path_to_main()}/ExcelFiles/TR/TR5_O2_FEV1_FEF2575_with_idx_28092025.xlsx", index=False)

# Demographics

In [4]:
def get_study_demographics(df):
    # df["BMI"] = df["Weight"] / (df["Height"] / 100) ** 2
    # print(f"Mean BMI: {df['BMI'].mean():.1f} ± {df['BMI'].std():.1f}")
    print(f"Mean Age: {df['Age'].mean():.1f} ± {df['Age'].std():.1f}")
    n_female = (df["Sex"] == "Female").sum()
    print(f"Female: {n_female} ({n_female/len(df)*100:.1f}%)")

    n_real = (
        df.ID.nunique()
        - df.groupby("ID").apply(lambda df: df["FEV1 % Predicted"].max()).isna().sum()
    )
    ppfev1 = df.groupby("ID").apply(lambda df: df["FEV1 % Predicted"].max())
    n_id_sup_90 = len(ppfev1[ppfev1 >= 90])
    prct_id_sup_90 = n_id_sup_90 / n_real * 100
    n_id_70_90 = len(ppfev1[(ppfev1 >= 70) & (ppfev1 < 90)])
    prct_id_70_90 = n_id_70_90 / n_real * 100
    n_id_40_70 = len(ppfev1[(ppfev1 >= 40) & (ppfev1 < 70)])
    prct_id_40_70 = n_id_40_70 / n_real * 100
    n_id_inf_40 = len(ppfev1[ppfev1 < 40])
    prct_id_inf_40 = n_id_inf_40 / n_real * 100

    print(
        f"ppFEV1: {df['FEV1 % Predicted'].mean():.1f}% ± {df['FEV1 % Predicted'].std():.1f}%"
    )
    print("ppFEV1 subgrouping")
    print(f"  >=90%: {n_id_sup_90} ({prct_id_sup_90:.0f}%)")
    print(f"  70-89%: {n_id_70_90} ({prct_id_70_90:.0f}%)")
    print(f"  40-69%: {n_id_40_70} ({prct_id_40_70:.0f}%)")
    print(f"  <40%: {n_id_inf_40} ({prct_id_inf_40:.0f}%)")
    print(f"  Total: {n_real}")
    return -1


get_study_demographics(df)

Mean Age: 65.7 ± 9.4
Female: 2865 (56.1%)
ppFEV1: 87.4% ± 18.0%
ppFEV1 subgrouping
  >=90%: 2421 (47%)
  70-89%: 1899 (37%)
  40-69%: 719 (14%)
  <40%: 66 (1%)
  Total: 5105


-1

In [5]:
df.columns

Index(['ID', 'UID', 'Age', 'Sex', 'Height', 'Health', 'Asthma', 'Bronchitis',
       'Smoke daily', 'Cigarettes number', 'Smoke years',
       'Respiratory infection 3W', 'Cough daily', 'Cough sputum',
       'Chest wheezing', 'Short winded walking fast', 'Short winded resting',
       'O2 Saturation', 'FEV1', 'FEF2575', 'Predicted FEV1',
       'Healthy O2 Saturation', 'FEV1 % Predicted', 'O2 Saturation % Healthy'],
      dtype='object')

In [127]:
disease = [
    "Asthma",
    "Bronchitis",
    "Respiratory infection 3W",
]
habits = [
    "Smoke daily",
    "Cigarettes number",
    "Smoke years",
]
qof = [
    "Health",
    "Cough daily",
    "Cough sputum",
    "Chest wheezing",
    "Short winded walking fast",
    "Short winded resting",
]
cols = disease + habits + qof
for col in disease:
    n = (df[col] == 1).sum()
    ntotal = len(df) - df[col].isna().sum()
    print(f"{col}: {n} ({n/ntotal*100:.1f}%), {ntotal} participants")
# for col in habits:
# n = (df[col] == 1).sum()
# ntotal = len(df) - df[col].isna().sum()
# print(f"{col}: {n} ({n/ntotal*100:.1f}%), {ntotal} participants")
# for col in qof:
#     print(df[col].value_counts())

Asthma: 467 (9.3%), 5002 participants
Bronchitis: 266 (5.3%), 4974 participants
Respiratory infection 3W: 751 (16.4%), 4579 participants


In [119]:
df["Asthma"].isna().sum()

103

In [128]:
len(df)

5105

# O2-FEV1 analysis

In [72]:
def plot_o2_fev_with_displots(O2_FEV1, x, y, title, rangex, rangey):

    opacity_scatter = 0.6
    opacity_displot = 1

    fig = make_subplots(
        rows=2,
        cols=2,
        shared_xaxes=True,
        shared_yaxes=True,
        column_widths=[0.8, 0.2],
        row_heights=[0.3, 0.7],
        vertical_spacing=0.02,
        horizontal_spacing=0.005,
    )

    x_val = O2_FEV1[x]
    y_val = O2_FEV1[y]

    # Add scatter plot
    fig.add_trace(
        go.Scatter(
            x=x_val,
            y=y_val,
            mode="markers",
            # name="Stable",
            marker=dict(
                size=5,
                color=ph.get_stable_color(opacity_scatter),
                line=dict(width=0.2, color="DarkSlateGrey"),
            ),
        ),
        row=2,
        col=1,
    )
    fig.update_xaxes(range=rangex, row=2, col=1)
    fig.update_yaxes(range=rangey, row=2, col=1)

    # Add displot for x
    if np.mean(x_val) > 4.5:
        # x_bins = dict(start=15, end=110, size=5)
        x_bins = dict(start=rangex[0], end=rangex[1], size=5)
    else:
        x_bins = dict(start=0.4, end=4.5, size=0.2)

    fig.add_trace(
        go.Histogram(
            x=x_val,
            histnorm="probability",
            # nbinsx=bins_n_stable,
            xbins=x_bins,
            marker=dict(color=ph.get_stable_color(opacity_displot)),
        ),
        row=1,
        col=1,
    )
    y_bins = dict(start=np.floor(rangey[0] * 0.9), end=np.ceil(rangey[1] * 1.1), size=1)
    fig.add_trace(
        go.Histogram(
            y=y_val,
            histnorm="probability",
            # Number of bins automatically set is good enough because O2 saturation is a dicsrete variable with a small values span
            ybins=y_bins,
            marker=dict(color=ph.get_stable_color(opacity_displot)),
        ),
        row=2,
        col=2,
    )

    # fig.update_layout(barmode="overlay")
    fig.update_xaxes(title_text=x, row=2, col=1)
    fig.update_yaxes(title_text=y, row=2, col=1)

    fig.update_layout(height=600, width=1300, title=title)

    return fig

In [82]:
title = f"Tromso T5 study, {df.shape[0]} entries"
xcol = "FEV1 % Predicted"
rangex = [
    np.floor(min(df[xcol].min(), dfBR[xcol].min())) * 0.9,
    np.ceil(max(df[xcol].max(), dfBR[xcol].max())) * 1.05,
]
ycol = "O2 Saturation"
rangey = [
    np.floor(min(df[ycol].min(), dfBR[ycol].min())) * 0.99,
    np.ceil(max(df[ycol].max(), dfBR[ycol].max())) * 1.01,
]
print(rangex, rangey)
fig = plot_o2_fev_with_displots(
    df, "FEV1 % Predicted", "O2 Saturation", title, rangex, rangey
)
fig.write_image(f"{dh.get_path_to_main()}/PlotsCrossStudies/{title}.pdf")
# fig.show()

[4.5, 156.45000000000002] [74.25, 101.0]


In [ ]:
title = f"Project Breathe study, {dfBR.shape[0]} entries"
fig = plot_o2_fev_with_displots(
    dfBR, "FEV1 % Predicted", "O2 Saturation", title, rangex, rangey
)
fig.write_image(f"{dh.get_path_to_main()}/PlotsCrossStudies/{title}.pdf")
# fig.show()

In [86]:
df.columns

Index(['ID', 'UID', 'Age', 'Sex', 'Height', 'Health', 'Asthma', 'Bronchitis',
       'Smoke daily', 'Cigarettes number', 'Smoke years',
       'Respiratory infection 3W', 'Cough daily', 'Cough sputum',
       'Chest wheezing', 'Short winded walking fast', 'Short winded resting',
       'O2 Saturation', 'FEV1', 'FEF2575', 'Predicted FEV1',
       'Healthy O2 Saturation', 'FEV1 % Predicted', 'O2 Saturation % Healthy'],
      dtype='object')